# TitanAgent Phase 2: 既存小型日本語 LLM ＋ Titans (MAC) 1本道チャット試作

基礎実験（Exp 1〜3.5）の結論を受け、**日本語が話せる小型オープン LLM（`Qwen/Qwen2.5-0.5B-Instruct`）を凍結し、Titans の MAC（Memory as a Context）長期記憶モジュールを接続する**試作プロトタイプです。

### このノートブックでできること（Kaggle T4 / Google Colab 推奨・約 20〜30 分）
1. **Titans Neural Memory（テスト時勾配更新 $M_t$）＋ MAC アダプタ**を `Qwen2.5-0.5B-Instruct` に接続
2. **2 段階 SFT（Exp 3.5 知見反映）**: 前提記憶・雑談ディストラクタ・前提上書き（`更新`）を Titans 記憶だけに流し込み、MAC メモリから思い出して `<think>` リハーサル付きで日本語応答する訓練を実施
3. **日本語記憶・上書き評価**（`memory_recall_rate` / `overwrite_rate` / `stale_rate`）
4. **Gradio 1 本道チャット UI（共有リンク）** をそのまま起動し、ブラウザから日本語で会話しながら Titans 記憶の更新（Surprise / 忘却ゲート / 圧縮ターン数）をリアルタイムに体感

> **重要**: Kaggle で実行する場合は **Accelerator = `GPU T4 x2`（または `P100`）**、**Internet = `On`** にしてください。

In [ ]:
# 1. 環境確認＆必要ライブラリ導入
import os, sys, subprocess, torch

IN_COLAB = 'google.colab' in sys.modules
IN_KAGGLE = os.path.exists('/kaggle/working')
print(f'Environment: Colab={IN_COLAB}, Kaggle={IN_KAGGLE}')
print(f'PyTorch: {torch.__version__}, CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('⚠️ WARNING: GPU が無効です！設定で GPU を ON にしてください。')

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'transformers>=4.45.0', 'accelerate', 'gradio>=4.40.0'])


In [ ]:
# 2. ハイパーパラメータ設定
from pathlib import Path

MODEL_NAME = 'Qwen/Qwen2.5-0.5B-Instruct'  # 余裕があれば 'Qwen/Qwen2.5-1.5B-Instruct' も可
MEM_DIM = 256
NUM_PERSIST = 4
NUM_LONGTERM = 8
TRAIN_STEPS = 200      # T4 で約 15〜25 分目安（まず動作確認なら 60 でも可）
GRAD_ACCUM = 4
LR = 1e-3
EVAL_EPISODES = 10
MAX_ACTIVE_TURNS = 3   # これを超えた古いターンは Titans Neural Memory のみに圧縮
LAUNCH_GRADIO = True   # Kaggle Save & Run All (Commit) 時は False 推奨
SEED = 42

OUTPUT_DIR = Path('/kaggle/working/checkpoints/phase2_mac_adapter') if IN_KAGGLE else Path('checkpoints/phase2_mac_adapter')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('OUTPUT_DIR =', OUTPUT_DIR)

In [ ]:
# 3. Titans (MAC) Neural Memory Adapter 定義
from dataclasses import asdict, dataclass
import math, random, re, time, json
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer

torch.manual_seed(SEED)
random.seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

@dataclass
class TitansMemoryConfig:
    hidden_size: int = 896
    mem_dim: int = 256
    num_persist_mem_tokens: int = 4
    num_longterm_mem_tokens: int = 8
    max_grad_norm: float = 1.0
    base_lr_scale: float = 0.15
    max_active_turns: int = 3

@dataclass
class TitansMemoryState:
    w1: torch.Tensor
    w2: torch.Tensor
    s1: torch.Tensor
    s2: torch.Tensor
    num_updates: int = 0
    last_surprise: float = 0.0
    last_forget_rate: float = 0.0

    def detach_(self):
        self.w1 = self.w1.detach(); self.w2 = self.w2.detach()
        self.s1 = self.s1.detach(); self.s2 = self.s2.detach()
        return self

class TitansMACAdapter(nn.Module):
    def __init__(self, cfg: TitansMemoryConfig):
        super().__init__()
        self.cfg = cfg
        h, d = cfg.hidden_size, cfg.mem_dim
        self.in_norm = nn.LayerNorm(h)
        self.k_proj = nn.Linear(h, d, bias=False)
        self.v_proj = nn.Linear(h, d, bias=False)
        self.q_proj = nn.Linear(h, d, bias=False)
        self.gate_proj = nn.Linear(h, 3)
        nn.init.zeros_(self.gate_proj.weight)
        with torch.no_grad():
            self.gate_proj.bias.copy_(torch.tensor([-3.0, 0.0, -1.0]))
        self.w1_init = nn.Parameter(torch.empty(d, d))
        self.w2_init = nn.Parameter(torch.empty(d, d))
        nn.init.normal_(self.w1_init, 0.0, 1.0 / math.sqrt(d))
        nn.init.normal_(self.w2_init, 0.0, 1.0 / math.sqrt(d))
        self.persist_mem = nn.Parameter(torch.randn(1, cfg.num_persist_mem_tokens, h) * 0.02)
        self.read_queries = nn.Parameter(torch.randn(1, cfg.num_longterm_mem_tokens, d) * 0.02)
        self.out_norm = nn.LayerNorm(d)
        self.out_proj = nn.Linear(d, h, bias=False)
        self.out_scale = nn.Parameter(torch.tensor(0.1))

    def init_state(self, batch_size: int, dev: torch.device) -> TitansMemoryState:
        w1 = self.w1_init.float().unsqueeze(0).expand(batch_size, -1, -1).clone()
        w2 = self.w2_init.float().unsqueeze(0).expand(batch_size, -1, -1).clone()
        return TitansMemoryState(w1=w1, w2=w2, s1=torch.zeros_like(w1), s2=torch.zeros_like(w2))

    def _mem_fwd(self, x, w1, w2):
        z = torch.bmm(x, w1)
        sig = torch.sigmoid(z)
        h = z * sig
        return torch.bmm(h, w2), z, sig, h

    def update_memory(self, segment_embeds, state: TitansMemoryState):
        x = self.in_norm(segment_embeds.float())
        k = F.normalize(self.k_proj(x), dim=-1)
        v = self.v_proj(x)
        gates = self.gate_proj(x.mean(dim=1))
        alpha = torch.sigmoid(gates[:, 0]).view(-1, 1, 1)
        eta = torch.sigmoid(gates[:, 1]).view(-1, 1, 1)
        theta = F.softplus(gates[:, 2]).view(-1, 1, 1) * self.cfg.base_lr_scale
        pred, z, sig, h = self._mem_fwd(k, state.w1, state.w2)
        diff = pred - v
        surprise = float((0.5 * diff.pow(2).mean()).detach().item())
        err = diff / max(1, k.shape[1])
        dw2 = torch.bmm(h.transpose(1, 2), err)
        dh = torch.bmm(err, state.w2.transpose(1, 2))
        dz = dh * (sig * (1.0 + z * (1.0 - sig)))
        dw1 = torch.bmm(k.transpose(1, 2), dz)
        max_n = self.cfg.max_grad_norm
        dw1 = dw1 * (max_n / dw1.flatten(1).norm(dim=1).view(-1, 1, 1).clamp_min(1e-6)).clamp(max=1.0)
        dw2 = dw2 * (max_n / dw2.flatten(1).norm(dim=1).view(-1, 1, 1).clamp_min(1e-6)).clamp(max=1.0)
        ns1 = eta * state.s1 - theta * dw1
        ns2 = eta * state.s2 - theta * dw2
        nw1 = (1.0 - alpha) * state.w1 + ns1
        nw2 = (1.0 - alpha) * state.w2 + ns2
        fg = float(alpha.mean().detach().item())
        return TitansMemoryState(nw1, nw2, ns1, ns2, state.num_updates + 1, surprise, fg), {'surprise': surprise, 'forget_rate': fg}

    def retrieve_prefix(self, prompt_embeds, state: TitansMemoryState, out_dtype):
        bsz = prompt_embeds.shape[0]
        pq = self.q_proj(self.in_norm(prompt_embeds.float()).mean(dim=1, keepdim=True))
        queries = F.normalize(self.read_queries.float().expand(bsz, -1, -1) + pq, dim=-1)
        ret, _, _, _ = self._mem_fwd(queries, state.w1, state.w2)
        ret = self.out_proj(self.out_norm(ret)) * self.out_scale
        persist = self.persist_mem.float().expand(bsz, -1, -1)
        return torch.cat([persist, ret], dim=1).to(dtype=out_dtype)

class TitansMACLLM(nn.Module):
    def __init__(self, base_model, mem_cfg: TitansMemoryConfig):
        super().__init__()
        self.base_model = base_model
        for p in self.base_model.parameters():
            p.requires_grad = False
        self.mem_cfg = mem_cfg
        self.adapter = TitansMACAdapter(mem_cfg)

    @property
    def prefix_len(self):
        return self.mem_cfg.num_persist_mem_tokens + self.mem_cfg.num_longterm_mem_tokens

    def embed_ids(self, ids):
        return self.base_model.get_input_embeddings()(ids)

    def memorize_segment(self, seg_ids, state):
        with torch.no_grad():
            emb = self.embed_ids(seg_ids)
        return self.adapter.update_memory(emb, state)

    def forward_with_memory(self, mem_segs, target_ids, labels):
        state = self.adapter.init_state(target_ids.shape[0], target_ids.device)
        stats = {'surprise': 0.0, 'forget_rate': 0.0}
        for seg in mem_segs:
            state, stats = self.memorize_segment(seg, state)
        with torch.no_grad():
            t_emb = self.embed_ids(target_ids)
        prefix = self.adapter.retrieve_prefix(t_emb, state, t_emb.dtype)
        out = self.base_model(inputs_embeds=torch.cat([prefix, t_emb], dim=1))
        shift_logits = out.logits[:, self.prefix_len : -1, :].contiguous()
        shift_labels = labels[:, 1:].contiguous()
        loss = F.cross_entropy(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1), ignore_index=-100)
        return loss, state, stats

    @torch.no_grad()
    def generate_with_memory(self, prompt_ids, state, eos_id, max_new_tokens=160, temperature=0.0):
        p_emb = self.embed_ids(torch.tensor([prompt_ids], dtype=torch.long, device=device))
        cur = torch.cat([self.adapter.retrieve_prefix(p_emb, state, p_emb.dtype), p_emb], dim=1)
        gen = []
        for _ in range(max_new_tokens):
            logits = self.base_model(inputs_embeds=cur).logits[:, -1, :]
            if temperature <= 1e-5:
                nid = int(torch.argmax(logits, dim=-1).item())
            else:
                probs = F.softmax(logits / temperature, dim=-1)
                nid = int(torch.multinomial(probs, 1).item())
            gen.append(nid)
            if nid == eos_id:
                break
            cur = torch.cat([cur, self.embed_ids(torch.tensor([[nid]], dtype=torch.long, device=device))], dim=1)
        return gen

print('Loading base Japanese LLM:', MODEL_NAME)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.float16 if device.type == 'cuda' else torch.float32, trust_remote_code=True
).to(device)
base_model.eval()

mem_cfg = TitansMemoryConfig(hidden_size=int(base_model.config.hidden_size), mem_dim=MEM_DIM, num_persist_mem_tokens=NUM_PERSIST, num_longterm_mem_tokens=NUM_LONGTERM, max_active_turns=MAX_ACTIVE_TURNS)
wrapper = TitansMACLLM(base_model, mem_cfg).to(device)
print(f'Trainable MAC Adapter params: {sum(p.numel() for p in wrapper.adapter.parameters())/1e6:.2f}M')

In [ ]:
# 4. 日本語 1本道記憶・上書きデータ生成器 ＆ 2段階 SFT 学習
NAMES = ['ケンタ', 'ユウキ', 'ミサキ', 'ハルカ', 'タクミ', 'アオイ', 'リョウ', 'サクラ']
THEMES = ['英語学習', 'アプリ開発', '副業ブログ', '資格勉強', '運動習慣', 'AI研究', 'デザイン学習', '自動化ツール作り']
GOALS = ['毎日続けること', '3か月以内に形にすること', '基礎を固めること', '小さく公開すること', '収入につなげること']
CONSTRAINTS = ['平日は30分だけ', '週末にまとめて作業', '予算は無料枠のみ', '夜しか時間がない', 'スマホ中心で進めたい']
STYLES = ['簡潔に要点だけ', '丁寧にステップ順で', '具体例を交えて', '背中を押す感じで']
DISTRACTORS = [
    'ユーザー: 今日は天気がいいですね。一息ついています。\nアシスタント: 無理のないペースでリフレッシュしてください。',
    'ユーザー: コーヒーを淹れて作業環境を整えました。\nアシスタント: 準備が整うと集中しやすいですね。続けましょう。',
    'ユーザー: 別の用事を片付けてきました。\nアシスタント: おかえりなさい！また続きを話しましょう。',
]

def gen_episode(rng, stage=2, force_overwrite=None):
    latest = {'名前': rng.choice(NAMES), 'テーマ': rng.choice(THEMES), '目標': rng.choice(GOALS), '制約': rng.choice(CONSTRAINTS), '好み': rng.choice(STYLES)}
    stale = {}
    segs = [
        f"ユーザー: はじめまして、{latest['名前']}です。取り組んでいるのは{latest['テーマ']}、目標は「{latest['目標']}」、制約は「{latest['制約']}」です。アドバイスは{latest['好み']}お願いします。\n"
        f"アシスタント: <think>記憶: 名前={latest['名前']} / テーマ={latest['テーマ']} / 目標={latest['目標']} / 制約={latest['制約']}</think>\n"
        f"{latest['名前']}さん、よろしくお願いします！"
    ]
    do_ov = (False if stage == 1 else rng.random() < 0.55) if force_overwrite is None else force_overwrite
    for _ in range(rng.randint(0, 1) if stage == 1 else rng.randint(1, 3)):
        segs.append(rng.choice(DISTRACTORS))
    if do_ov:
        slot = rng.choice(['制約', '目標', 'テーマ'])
        old_v = latest[slot]
        pool = CONSTRAINTS if slot == '制約' else (GOALS if slot == '目標' else THEMES)
        new_v = rng.choice([x for x in pool if x != old_v])
        stale[slot] = old_v; latest[slot] = new_v
        segs.append(f"ユーザー: 状況が変わりました。{slot}を「{new_v}」に変更します。\nアシスタント: <think>更新記憶: {slot}={new_v}</think>\n了解しました！{slot}を「{new_v}」に更新します。")
        if stage == 2:
            segs.append(rng.choice(DISTRACTORS))
    prompt = 'ユーザー: 前に伝えた私の名前と、今のテーマ・目標・制約を踏まえて、次の一歩を提案してくれますか？\nアシスタント: '
    target = (
        f"<think>記憶リハーサル: 名前={latest['名前']} | テーマ={latest['テーマ']} | 目標={latest['目標']} | 制約={latest['制約']}</think>\n"
        f"{latest['名前']}さん、現在の前提（テーマ: {latest['テーマ']} / 目標: {latest['目標']} / 制約: {latest['制約']}）に合わせて{latest['好み']}提案します。"
        f"まずは「{latest['制約']}」の中で{latest['テーマ']}を小さく1回試しましょう。"
    )
    return segs, prompt, target, latest, stale, do_ov

rng = random.Random(SEED)
optimizer = torch.optim.AdamW(wrapper.adapter.parameters(), lr=LR, weight_decay=0.01)
stage1_steps = max(1, int(TRAIN_STEPS * 0.30))
wrapper.adapter.train()
t0 = time.time()

for step in range(1, TRAIN_STEPS + 1):
    stage = 1 if step <= stage1_steps else 2
    prog = step / max(1, TRAIN_STEPS)
    cur_lr = LR * (0.1 + 0.9 * 0.5 * (1.0 + math.cos(math.pi * prog)))
    for pg in optimizer.param_groups: pg['lr'] = cur_lr
    optimizer.zero_grad(set_to_none=True)
    aloss, asurp = 0.0, 0.0
    for _ in range(GRAD_ACCUM):
        segs, prompt, target, _, _, _ = gen_episode(rng, stage=stage)
        mem_t = [torch.tensor([tokenizer.encode(s, add_special_tokens=False)], dtype=torch.long, device=device) for s in segs]
        p_ids = tokenizer.encode(prompt, add_special_tokens=False)
        r_ids = tokenizer.encode(target, add_special_tokens=False) + [tokenizer.eos_token_id]
        full_t = torch.tensor([p_ids + r_ids], dtype=torch.long, device=device)
        lbl_t = torch.tensor([[-100] * len(p_ids) + r_ids], dtype=torch.long, device=device)
        loss, _, st = wrapper.forward_with_memory(mem_t, full_t, lbl_t)
        (loss / GRAD_ACCUM).backward()
        aloss += float(loss.item()) / GRAD_ACCUM; asurp += st['surprise'] / GRAD_ACCUM
    torch.nn.utils.clip_grad_norm_(wrapper.adapter.parameters(), 1.0)
    optimizer.step()
    if step == 1 or step % 20 == 0 or step == TRAIN_STEPS:
        print(f'  [MAC | S{stage}] step {step:4d}/{TRAIN_STEPS} | loss {aloss:.4f} | surprise {asurp:.4f} | {time.time()-t0:.1f}s')

ckpt_path = OUTPUT_DIR / 'titans_mac_adapter.pt'
torch.save({'mem_cfg': asdict(mem_cfg), 'adapter_state_dict': wrapper.adapter.state_dict()}, ckpt_path)
print('Saved MAC adapter to:', ckpt_path)

In [ ]:
# 5. 日本語記憶・上書き評価（短期プロンプトに過去ターンを含めず Titans 記憶のみで回答）
wrapper.adapter.eval()
eval_rng = random.Random(SEED + 999)
recalls, ov_ok, ov_tot, stale_cnt = 0.0, 0, 0, 0

for i in range(EVAL_EPISODES):
    segs, prompt, _, latest, stale, do_ov = gen_episode(eval_rng, stage=2, force_overwrite=(i % 2 == 1))
    st = wrapper.adapter.init_state(1, device)
    for s in segs:
        st, _ = wrapper.memorize_segment(torch.tensor([tokenizer.encode(s, add_special_tokens=False)], dtype=torch.long, device=device), st)
    gen_ids = wrapper.generate_with_memory(tokenizer.encode(prompt, add_special_tokens=False), st, tokenizer.eos_token_id, max_new_tokens=140, temperature=0.0)
    reply = tokenizer.decode(gen_ids, skip_special_tokens=True)
    recalls += sum(1 for k in ['名前', 'テーマ', '目標', '制約'] if latest[k] in reply) / 4.0
    if do_ov:
        ov_tot += 1
        if all(latest[k] in reply for k in stale): ov_ok += 1
        if any(old_v in reply and latest[k] not in reply for k, old_v in stale.items()): stale_cnt += 1
    if i == 1:
        print('--- Sample Overwrite Eval ---')
        print('Latest:', latest, '| Stale:', stale)
        print('Reply :', reply)

print(f'\n=== Phase 2 Eval ({EVAL_EPISODES} eps) ===')
print(f'memory_recall_rate = {recalls/EVAL_EPISODES:.3f}')
print(f'overwrite_rate     = {ov_ok/max(1,ov_tot):.3f}')
print(f'stale_rate         = {stale_cnt/max(1,ov_tot):.3f}')

In [ ]:
# 6. Gradio 1本道日本語チャット UI 起動（インタラクティブ実行時のみ）
if LAUNCH_GRADIO:
    import gradio as gr
    THINK_RE = re.compile(r'<think>.*?</think>\s*', flags=re.DOTALL)
    mem_state = wrapper.adapter.init_state(1, device)
    active_turns, full_history = [], []
    archived_turns = [0]

    def chat_step(user_msg, show_think, temp):
        nonlocal mem_state
        if not user_msg.strip():
            return '', [], ''
        lines = ['システム: あなたは1本道の長期会話で育つパーソナルAIです。重要な前提を<think>...</think>で確認してから自然な日本語で答えてください。']
        for idx, (u, a) in enumerate(active_turns):
            a_clean = THINK_RE.sub('', a).strip() if idx < len(active_turns) - 1 else a
            lines.append(f'ユーザー: {u}\nアシスタント: {a_clean}')
        lines.append(f'ユーザー: {user_msg}\nアシスタント: ')
        p_ids = tokenizer.encode('\n'.join(lines), add_special_tokens=False)
        gen_ids = wrapper.generate_with_memory(p_ids, mem_state, tokenizer.eos_token_id, max_new_tokens=160, temperature=float(temp))
        reply = tokenizer.decode(gen_ids, skip_special_tokens=True).strip()
        seg = f'ユーザー: {user_msg}\nアシスタント: {reply}'
        mem_state, _ = wrapper.memorize_segment(torch.tensor([tokenizer.encode(seg, add_special_tokens=False)], dtype=torch.long, device=device), mem_state)
        mem_state.detach_()
        full_history.append((user_msg, reply))
        active_turns.append((user_msg, reply))
        while len(active_turns) > MAX_ACTIVE_TURNS:
            active_turns.pop(0)
            archived_turns[0] += 1
        msgs = []
        for u, a in full_history:
            msgs.append({'role': 'user', 'content': u})
            msgs.append({'role': 'assistant', 'content': a if show_think else (THINK_RE.sub('', a).strip() or a)})
        mon = (
            f'### Titans 長期記憶モニター\n'
            f'- 総ターン数: `{len(full_history)}` | 短期枠: `{len(active_turns)}/{MAX_ACTIVE_TURNS}` | Titans圧縮済: `{archived_turns[0]}`\n'
            f'- 更新回数: `{mem_state.num_updates}` | Surprise: `{mem_state.last_surprise:.5f}` | 忘却α: `{mem_state.last_forget_rate:.5f}`'
        )
        return '', msgs, mon

    with gr.Blocks(title='TitanAgent Phase 2 Chat') as demo:
        gr.Markdown('# TitanAgent Phase 2: 1本道日本語チャット（Titans MAC）')
        chatbot = gr.Chatbot(type='messages', height=420)
        monitor = gr.Markdown('### Titans 長期記憶モニター\n待機中')
        with gr.Row():
            box = gr.Textbox(placeholder='日本語で話しかけてください...', scale=4)
            btn = gr.Button('送信', variant='primary', scale=1)
        with gr.Row():
            show_think = gr.Checkbox(value=True, label='<think>を表示')
            temp = gr.Slider(0.0, 1.0, value=0.4, step=0.1, label='Temperature')
        btn.click(chat_step, [box, show_think, temp], [box, chatbot, monitor])
        box.submit(chat_step, [box, show_think, temp], [box, chatbot, monitor])
    demo.launch(share=True)
else:
    print('LAUNCH_GRADIO=False のため UI 起動をスキップしました。')